# Notebook 5: Feedforward and Recurrent Neural Networks

This notebook introduces you to some basic principles of simple neural network models.


In [ ]:
# load matplotlib inline mode
%matplotlib inline

# import some useful libraries
import sys
import numpy as np                # numerical analysis linear algebra
import matplotlib as mpl
import matplotlib.pyplot as plt   # plotting
from scipy import signal
from scipy import linalg

# set some style options
mpl.rcParams['image.origin'] = 'lower'
mpl.rcParams['image.aspect'] = 'auto'
mpl.rcParams['image.cmap'] = 'jet'

## Rate models

We're going to build our network models using very simplified neurons. This allows us to illustrate some important principles of biological neural networks without getting bogged down in the details or in computationally expensive simulations. 

1. Each neuron will be defined by a single state variable $\nu \in \mathbb{R}$, which we define as the rate of firing relative to some baseline. This allows $\nu$ to take on negative numbers.
2. $\nu$ depends on a weighted sum of its synaptic inputs. Because each neuron can receive synapses from many other neurons, this input is represented as a vector  
$\mathbf{u} \in \mathbb{R}^k$. The weights of the synapses from the $k$ inputs are also a vector, represented as $\mathbf{w} \in \mathbb{R}^k$.

<img src="https://meliza.org/public/courseware/comp-neurosci/images/l13_d7.1.png" alt="feedforward model" style="width: 400px;"/>

We use a weighted linear sum to combine the inputs because the total synaptic current entering the cell, $I$, is effectively a linear sum of the individual synaptic currents, $\mathbf{w}\mathbf{u}$. Each input is multiplied by the corresponding weight and then summed.

The inputs sum linearly, but the relationship between total input current $I$ and the output rate $\nu$ may not be linear. Even though we're allowing $\nu$ to be unbounded and correcting it relative to a baseline, real neurons have minimum and maximum firing rates. We represent this nonlinear relationship by a nonlinear function $\nu = F(I)$. We might make $F(I)$ look like a sigmoid function to reflect how cells don't spike at all when hyperpolarized and saturate at some maximum rate when depolarized strongly, but there are other popular choices.

### Membrane dynamics

We now have to make a decision about time. In an *artificial neural network* (ANN), there either is no time, or it moves in large, discrete steps. This allows us to treat the input-output relationship of the neuron as **instantaneous**:

$$
\nu(t) = F(\mathbf{w} \cdot \mathbf{u}(t)).
$$

That is, the rate at any moment in time $\nu(t)$ is only a function of whatever stimulus is present, $\mathbf{u}(t)$. As we saw previously, we can introduce a memory for the *past stimulus* by delay-embedding $\mathbf{u}(t)$. We can also give the neuron a memory by introducing *dynamics*.

A simple first choice for a dynamical model is based on a leaky integrator:

$$
\tau \frac{\mathrm{d}\nu(t)}{\mathrm{d}t} = - \nu(t) + F(\mathbf{w} \cdot \mathbf{u}(t))
$$

This equation states that in the absence of any input, the firing rate tends to decay to zero (baseline) with a time constant of $\tau$, and external synaptic currents perturb the neuron from this resting state. The neuron's current state $\nu(t)$ therefore acts as a memory of the past inputs, with more distant memories having exponentially less of an effect.

#### Example

Let's build an example model visual neuron that integrates 21 inputs, each tuned to a direction between -180 and 180 degrees. The neuron has weights that give it a gaussian tuning curve centered at 0 degrees with a standard deviation of 30 degrees.   

In [ ]:
n_inputs = 21
directions = np.linspace(-180, 180, n_inputs)

def tuning_curve(preferred, width):
    return np.exp(-(directions - preferred)**2 / width**2)

weights = tuning_curve(0, 30)
fig, axes = plt.subplots(nrows=1, ncols=1, sharex=True, figsize=(3, 3))
axes.plot(directions, weights)
axes.set_xlabel("Theta (deg)")
axes.set_ylabel("Weight");

We'll start by simulating the response of this neuron to a pseudorandom sequence of moving bars at 10 ms intervals.

In [ ]:
np.random.seed(1024)
n_frames = 100
sparse_directions = np.random.randint(0, n_inputs, n_frames)
sparse_input = np.zeros((n_frames, n_inputs))
for frame, dir in enumerate(sparse_directions):
    sparse_input[frame, dir] = 1

fig, axes = plt.subplots(nrows=1, ncols=1, sharex=True, figsize=(12, 2))
axes.imshow(sparse_input.T, cmap="gray")
axes.set_xlabel("Frame")
axes.set_ylabel("Input")

To simulate the response, we need to integrate the differential equation $\tau \frac{\mathrm{d}\nu(t)}{\mathrm{d}t} = - \nu(t) + F(\mathbf{w} \cdot \mathbf{u}(t))$. To illustrate the basic principle of numerical integration, we'll use Euler's method:

$$
\nu(t+\Delta t) = \nu(t) + \frac{\mathrm{d}\nu(t)}{\mathrm{dt}} \Delta t
$$

We'll use a time constant of 20 ms and a time step of 1 ms.

In [ ]:
dt_step = 1    # interval between integration steps
dt_stim = 10  # interval between stimulus frames
n_step = int(n_frames * dt_stim / dt_step)
t_step = np.arange(0, n_step * dt_step, dt_step)
tau = 20      # time constant of the neuron's membrane
gain = 1.5    # gain of the sigmoidal nonlinearity
thresh = 3.0  # threshold of the sigmoidal nonlinearity

def sigmoid(x):
    """ Calculate the sigmoidal nonlinearity """
    return (1 + np.exp(-gain * (x - thresh)))**-1 - (1 + np.exp(gain * thresh))**-1

def dvdt(v, step):
    """ This function calculates dvdt given the current voltage and time """
    frame = int(step * dt_step // dt_stim)
    stim = sparse_input[frame]
    current = np.dot(stim, weights)
    return (sigmoid(current) - v) / tau

# Euler's method: for each step, calculate dvdt, multiply by step size, and add to previous value

# need an initial condition
v_prev = 0
# need an array to store the output
vout = np.zeros(n_step)
for step in range(n_step):
    dv = dvdt(v_prev, step) * dt_step
    v_prev += dv
    vout[step] = v_prev

In [ ]:
fig, axes = plt.subplots(nrows=2, ncols=1, sharex=True, figsize=(12, 4))
axes[0].imshow(sparse_input.T, extent=(0, n_frames * dt_stim, directions[0], directions[-1]), cmap="gray")
axes[0].set_ylabel("Theta (deg)")
axes[1].plot(t_step, vout)
axes[1].set_ylabel("Rate (a.u.)")
axes[1].set_xlabel("Time (ms)")

### Exercise 1

Simulate the response of the example neuron above to *gaussian white noise* with a mean of 0 and a standard deviation of 1. In white noise, the inputs are random and uncorrelated (hint: use `np.random.normal`). Use reverse correlation to verify that the receptive field of the neuron has the same direction tuning as the weights. You will probably need at least 10,000 frames to obtain a good estimate. What does the temporal profile of the receptive field look like? How does it compare to the time constant of the neuron?

NB: Pay close attention to time. The stimulus is on a different timebase than the response, so you'll either have to downsample the response or upsample the stimulus to compute the correlation. You can use linear algebra to compute the correlation ($\mathbf{\hat{w}} = \mathbf{u}^T\nu$) but you will need to delay-embed $\mathbf{u}$ first.

## Feedforward networks

We can use our rate model to build simple and complex networks. If our network structure can be cleanly divided into a set of layers, with each layer only receiving input from the previous layer, we have a **feedforward network**. The simplest feedforward network has a single input and a single output layer:

<img src="https://meliza.org/public/courseware/comp-neurosci/images/l13_d7.3.png" alt="feedforward network" style="width: 400px;"/>

This network differs from our single-neuron rate model only in having $n$ output neurons, each receiving inputs from the $k$ input neurons. The state of the output layer is now a vector $\mathbf{v} \in \mathbb{R}^n$, and the connection weights from the input layer are defined matrix $\mathbf{W}$ with $n$ rows and $k$ columns.

Each row in $\mathbf{W}$ is a $k$-dimensional vector corresponding to the weights of the input vector for one of the output neurons. Positive values in $\mathbf{W}$ indicate excitatory connections; negative values indicate inhibitory connections; and zero values indicate the absence of a connection. $\mathbf{W}$ may have many elements set to zero if the connectivity is sparse. 

In algebraic terms, $\mathbf{W}$ is a linear map from the input space $\mathbf{u} \in \mathbb{R}^k$ to the output space $\mathbf{v} \in \mathbb{R}^n$.

The model for the $i$-th output neuron is specified using the appropriate row of weights from $\mathbf{W}$. For brevity, we will drop the $(t)$ notation for $\nu$ and $u$. 

$$
\tau \frac{\mathrm{d}\nu_i}{\mathrm{d}t} = - \nu_i + F\left(\sum_{j} W_{ij} u_j\right)
$$

In vector notation:

$$
\tau \frac{\mathrm{d}\mathbf{v}}{\mathrm{d}t} = - \mathbf{v} + F(\mathbf{W} \cdot \mathbf{u})
$$

Note that each neuron depends only on its inputs, so although there are potentially many ODEs in the system, they are **uncoupled**.

#### Associative nets

Hopefully this structure reminds you of the pen-and-paper networks we worked with in our first class. Each input has the potential to connect to any and all outputs

<img src="https://meliza.org/public/courseware/comp-neurosci/images/l13_s9.4.png" alt="associative net" style="width: 700px;"/>

### Computational properties of feedforward networks

Feedforward connectivity is an unrealistic picture of how biological networks are organized, but FF networks are still useful tools for understanding neural computations and for performing machine learning. The receptive field models we explored previously are effectively FF models in which $\mathbf{u}$ represents features of the visual or acoustic stimulus rather than the firing rates of our neurons' synaptic inputs. 

As we'll see in later weeks, FF networks constructed with many layers can approximate very complex nonlinear functions.

### Exercise 2

Build out the example single neuron from exercise 1 into a feedforward network with 10 output neurons. Each of the output neurons should have a gaussian tuning curve, but vary the preferred direction in equal steps from -90 to 90 degrees. When calculating $\frac{dv}{dt}$, use vector operations, not for loops. Plot the response of the population as a heatmap to the original sparse noise stimulus.

## Recurrent networks

We can make our FF network *recurrent* by allowing the neurons of the output layer to be connected to each other.

<img src="https://meliza.org/public/courseware/comp-neurosci/images/l13_d7.3b.png" alt="feedforward network" style="width: 400px;"/>

The connection weights within the $n$ neurons of the output layer are given by a matrix $\mathbf{M} \in \mathbb{R}^{n \times n}$. These currents sum linearly with the currents from the input layer, so the model now looks like:

$$
\tau \frac{\mathrm{d}\mathbf{v}}{\mathrm{d}t} = - \mathbf{v} + F(\mathbf{W} \cdot \mathbf{u} + \mathbf{M} \cdot \mathbf{v})
$$

Because the $\mathbf{W} \cdot \mathbf{u}$ term doesn't depend on the state of the system, it's common to replace it with a single vector $\mathbf{h} \in \mathbb{R}^n$ representing the extrinsic input to each of the $n$ neurons in the output layer:

$$
\tau \frac{\mathrm{d}\mathbf{v}}{\mathrm{d}t} = - \mathbf{v} + F(\mathbf{h} + \mathbf{M} \cdot \mathbf{v})
$$

In contrast to the feedforward network, the ODEs in a recurrent networks are **coupled** to each other. This opens up a whole realm of computationally interesting possibilities.


### Exercise 3

Add a feedback connectivity matrix $\mathbf{M}$ to the feedforward model. 

A. Set $\mathbf{M}$ to all zeros and verify that the model behaves the same as your feedforward network in Exercise 2.

B. Simulate the model with a sparse, random excitatory connections. Randomly choose 15 connections in $\mathbf{M}$ and set them to a random value between 1.0 and 10. Plot the connectivity matrix as a heatmap and plot the response to the example sparse stimulus.

C. Simulate a model with mutual inhibition. Connect all the neurons with negative preferred directions to each other with excitatory weights of 1.0. Connect all the neurons with positive preferred directions to each other with excitatory weights of 1.0. Connect all the positive-preferring neurons to all the negative-preferring neurons with inhibitory weights of 0.5. Plot the connectivity matrix as a heatmap and plot the response to the example sparse stimulus.

## Linear recurrent networks

Because of the principle of topological invariance, we can learn a lot about the dynamics of recurrent neural networks by linearizing them. Let's start by just making $F(I) = I$. Now the system of equations becomes:

$$
\tau \frac{\mathrm{d}\mathbf{v}}{\mathrm{d}t} = - \mathbf{v} + \mathbf{M} \cdot \mathbf{v} + \mathbf{h}
$$

Recall that an $n \times n$ matrix has $n$ eigenvectors that satisfy 

$$
\mathbf{M} \cdot \mathbf{e}_\mu = \lambda_\mu \mathbf{e}_\mu,
$$

and that the eigenvectors $\mathbf{e}_\mu$ are $n$-dimensional vectors where the linear map $M$ causes a scalar change of $\lambda_\mu$ in the magnitude of the vector but not a change in direction.

If $\mathbf{M}$ is symmetric (i.e., neuron $i$ is connected to neuron $j$ with the same weight that neuron $j$ is connected to neuron $i$), then all of the eigenvectors and eigenvalues will be real and orthonormal:

$$\mathbf{e}_\mu \cdot \mathbf{e}_\nu = \begin{cases} 1 & \mu = \nu \\ 0 & \mu \neq \nu \\ \end{cases}$$

Because of this, any $N$-element vector can be expressed as a weighted sum of the eigenvectors. This includes $\mathbf{v}$:

$$
\mathbf{v}(t) = \mathbf{c}(t) \cdot \mathbf{e} = \sum_\mu^n c_\mu(t) \mathbf{e}_\mu
$$


We can interpret $\mathbf{c}(t)$ as the activity of the neural population in *latent, mutually orthogonal dimensions*. This allows us to rewrite our model as follows:

\begin{align}
\tau \frac{\mathrm{d}\mathbf{v}}{\mathrm{d}t} & = - \mathbf{v} + \mathbf{M} \cdot \mathbf{v} +\mathbf{h} \\
\tau \sum_\mu^n \frac{\mathrm{d}c_\mu}{\mathrm{d}t}\mathbf{e}_\mu & = - \sum_\mu^n (1 - \lambda_\mu)c_\mu(t)\mathbf{e}_\mu + \mathbf{h}
\end{align}

Then, taking advantage of orthogonality, we can take the dot product on each side with $\mathbf{e}_\nu$ to get a solution for any eigenvector:

$$
\tau \frac{\mathrm{d}c_\nu}{\mathrm{d}t} = - (1 - \lambda_\nu)c_\nu(t) + \mathbf{e}_\nu \cdot \mathbf{h}
$$

The key result here is that each latent component $c_\nu(t)$ depends only on itself and on a weighted sum of the extrinsic inputs, $\mathbf{e}_\nu \cdot \mathbf{h}$. In other words, the system reduces to a set of **uncoupled**, first-order ODEs.

### Selective amplification

A useful consequence of being able to orthogonalize the connectivity matrix is that the rate of each eigenvector has a simple, analytical solution:

$$
c_\nu(t) = \frac{\mathbf{e}_\nu \cdot \mathbf{h}}{1 - \lambda_\nu}
\left(1 - \exp \left(- \frac{t(1 - \lambda_\nu)}{\tau}\right)\right) + 
c_\nu(0) \exp \left(- \frac{t(1 - \lambda_\nu)}{\tau}\right)
$$

The eigenvalues here have a similar interpretation as before:

- $\lambda_\nu > 1$: the system will diverge exponentially to infinity
- $\lambda_\nu < 1$: $c_\nu(t)$ will exponentially converge to the steady-state value $(\mathbf{e}_\nu \cdot \mathbf{h})/(1 - \lambda_\nu)$

Moreover, the steady state of the entire network is a weighted sum of the eigenstates:

$$
\mathbf{v}_\infty = \sum_\nu^N \frac{\mathbf{e}_\nu \cdot \mathbf{h}}{1 - \lambda_\nu} \mathbf{e}_\nu
$$

If one of the eigenvalues ($\lambda_1$) is close to 1, and all the others are relatively small, then the denominator $(1 - \lambda_1)$ will be close to zero, and the projection of the stimulus onto this eigenvector will dominate the response:

$$
\mathbf{v}_\infty \approx \frac{\mathbf{e}_1 \cdot \mathbf{h}}{1 - \lambda_1} \mathbf{e}_1
$$

For example, this figure from Carandini and Ringach (1997) compares the input and output of a linear recurrent network modeling visual orientation selectivity.

<img src="https://meliza.org/public/courseware/comp-neurosci/images/l13_d7.8.png" alt="selective amplification" style="width: 400px;"/>

The input is relatively noisy and untuned, but the network's eigenvector with the largest eigenvalue ($\lambda_1 = 0.9$) has a broad peak at around 0 degrees, and this selectively amplifies the projection of the input onto that eigenvector.

### Exercise 4

Compute the eigenvectors and eigenvalues of the connectivity matrix from network C in Exercise 3 (NB: if numpy gives you complex numbers even though your matrix is symmetric, use `.real` to ignore the imaginary components). Plot the distribution of the eigenvalues. Which eigenvectors will dominate the dynamics of the network?

See if you can replicate the result from Carandini and Ringach (1997) by giving the network a constant white noise input. You may need to play around with the connectivity matrix to create a dominant mode centered around 0 degrees.